# Data Preperation

In [46]:
import yfinance as yf
import pandas as pd
import numpy as np

In [47]:
ticker = '^GSPC'

In [48]:
data = yf.download(ticker, start='2015-01-01', end='2025-01-01')

/var/folders/q3/wwj9bvx95wdbjv6cnbfqkxm40000gn/T/ipykernel_27692/3354553798.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start='2015-01-01', end='2025-01-01')
[*********************100%***********************]  1 of 1 completed


In [49]:
print(data.columns)

MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])


In [50]:
data.columns = data.columns.droplevel(1)

In [51]:
print(data.columns)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')


In [52]:
# Daily return percentage
data['Return'] = data['Close'].pct_change()

# Moving averages
data['SMA_10'] = data['Close'].rolling(10).mean()
data['SMA_50'] = data['Close'].rolling(50).mean()

# RSI(Relative Strength Index) (14-day)

delta = data['Close'].diff()
gain = (delta.where(delta>0,0)).rolling(14).mean()
loss = (-delta.where(delta<0,0)).rolling(14).mean()
RS = gain/loss
data['RSI'] = 100 - (100/(1+RS))

# 10 day volatility
data['Volatility'] = data['Return'].rolling(10).std()

data.head()

Price,Close,High,Low,Open,Volume,Return,SMA_10,SMA_50,RSI,Volatility
Date,,,,,,,,,,
2015-01-02,2058.199951,2072.360107,2046.040039,2058.899902,2708700000,NaN,NaN,NaN,NaN,NaN
2015-01-05,2020.579956,2054.439941,2017.339966,2054.439941,3799120000,-0.018278,NaN,NaN,NaN,NaN
2015-01-06,2002.609985,2030.250000,1992.439941,2022.150024,4460110000,-0.008893,NaN,NaN,NaN,NaN
2015-01-07,2025.900024,2029.609985,2005.550049,2005.550049,3805480000,0.011630,NaN,NaN,NaN,NaN
2015-01-08,2062.139893,2064.080078,2030.609985,2030.609985,3934010000,0.017888,NaN,NaN,NaN,NaN


In [53]:
data = data.dropna()

data.shape

(2467, 10)

In [54]:
# Setting target to 0 or 1 (down or up the next day)
data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)

In [55]:
data.head()

Price,Close,High,Low,Open,Volume,Return,SMA_10,SMA_50,RSI,Volatility,Target
Date,,,,,,,,,,,
2015-03-16,2081.189941,2081.409912,2055.350098,2055.350098,3295600000,0.013534,2074.297986,2059.713201,40.901105,0.009999,0
2015-03-17,2074.280029,2080.590088,2065.080078,2080.590088,3221840000,-0.003320,2070.947986,2060.034802,39.784190,0.009968,1
2015-03-18,2099.500000,2106.850098,2061.229980,2072.840088,4128210000,0.012158,2071.044983,2061.613203,47.395975,0.010785,0
2015-03-19,2089.270020,2098.689941,2085.560059,2098.689941,3305220000,-0.004873,2069.867981,2063.346404,46.535642,0.010887,1
2015-03-20,2108.100098,2113.919922,2090.320068,2090.320068,5554120000,0.009013,2073.551990,2064.990405,47.942457,0.010093,0


In [56]:
output_path = "../data/sp500.csv"

data.to_csv(output_path, index=True)